[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyneuro/Chapter_Colabs/blob/main/Colab_E.ipynb)


# Set E — Synapses & Two‑Neuron Communication (LEGO–Colab)
**Author: Neural Engineering Laboratory, University of Missouri - Gregory Glickert, Khuram Choudhry, Ziao Chen, Satish S. Nair

**Run order:** E0 ▶ E1 ▶ … ▶ E9. If things desync: *Runtime ▶ Restart & run all*.  
This version uses **NetStim** (built‑in artificial cell) instead of `VecStim` to maximize Colab compatibility.

---
## Table of Contents
- [E0 — Starter](#e0)
- [E1 — Two neurons + AMPA‑like synapse](#e1)
- [E2 — Inhibitory (GABA_A‑like)](#e2)
- [E3 — EPSP vs IPSP (peaks & latencies)](#e3)
- [E4 — Exercise: excitatory vs inhibitory? (E_rev sweep)](#e4)
- [E5 — Temporal summation (two spikes, variable ISI)](#e5)
- [E6 — Spatial summation (proximal vs distal)](#e6)
- [E7 — Feedforward inhibition (E then I)](#e7)
- [E8 — Analysis helpers (latency, peak, area)](#e8)
- [E9 — Playground](#e9)
- [Reflection](#reflection)
---


## E0 — Starter (run once) <a id='e0'></a>

In [ ]:

!pip -q install neuron==8.2.4
try:
    from neuron import h, gui
except Exception:
    from neuron import h
from neuron.units import ms, mV
import numpy as np
import matplotlib.pyplot as plt
h.load_file('stdrun.hoc')

s0=h.Section(name='bootstrap')

def plot_vm(t_vec,traces,labels,title='',figsize=(7,3.3)):
    plt.figure(figsize=figsize)
    for tr,lb in zip(traces,labels):
        plt.plot(t_vec,tr,label=lb)
    plt.xlabel('Time (ms)'); plt.ylabel('Voltage (mV)'); plt.title(title)
    plt.grid(True,alpha=0.3);
    if len(labels)>1: plt.legend()
    plt.tight_layout(); plt.show()

t = h.Vector(); t.record(h._ref_t)

def mk_rec(sec):
    v=h.Vector(); v.record(sec(0.5)._ref_v); return v

print('E0 ready. Proceed to E1.')


## E1 — Two neurons + excitatory synapse (AMPA‑like) <a id='e1'></a>

In [ ]:

somaA=h.Section(name='somaA'); somaA.L=somaA.diam=20; somaA.Ra=100; somaA.cm=1; somaA.insert('hh')
somaB=h.Section(name='somaB'); somaB.L=somaB.diam=20; somaB.Ra=100; somaB.cm=1; somaB.insert('hh')

iclampA=h.IClamp(somaA(0.5)); iclampA.delay=5; iclampA.dur=2; iclampA.amp=0.4
syn=h.ExpSyn(somaB(0.5)); syn.tau=2.0; syn.e=0.0
nc=h.NetCon(somaA(0.5)._ref_v,syn,sec=somaA)
nc.threshold=-20; nc.delay=1.0; nc.weight[0]=0.005
vA=mk_rec(somaA); vB=mk_rec(somaB)
h.finitialize(-65*mV); h.continuerun(30*ms)
plot_vm(t,[vA,vB],["Neuron A","Neuron B"],title='E1: A→B excitatory (AMPA-like)')


## E2 — Switch to inhibitory synapse (GABA_A‑like) <a id='e2'></a>

In [ ]:

syn.e=-75.0; syn.tau=8.0; nc.weight[0]=0.01
h.finitialize(-65*mV); h.continuerun(40*ms)
plot_vm(t,[vA,vB],["Neuron A","Neuron B"],title='E2: A→B inhibitory (GABA_A-like)')


## E3 — EPSP vs IPSP contrast (peaks & latencies) <a id='e3'></a>

In [ ]:

import numpy as np

def run_trial(e_rev,tau,w,ttl):
    syn.e=e_rev; syn.tau=tau; nc.weight[0]=w
    h.finitialize(-65*mV); h.continuerun(35*ms)
    vnp=np.array(vB); tnp=np.array(t)
    peak=vnp.max(); tpk=tnp[vnp.argmax()]
    plot_vm(t,[vB],["Neuron B"],title=f"{ttl} | peak={peak:.1f} mV @ {tpk:.2f} ms")
    return peak,tpk
p1=run_trial(0.0,2.0,0.005,'E3: EPSP')
p2=run_trial(-75.0,8.0,0.01,'E3: IPSP')
print('EPSP:',p1,'IPSP:',p2)


## E4 — *Exercise*: excitatory vs inhibitory? (Reversal sweep) <a id='e4'></a>

In [ ]:

import numpy as np, matplotlib.pyplot as plt

def reversal_sweep(E_values=[-85,-75,-65,-55,-45,-35,-25,-15,-5,5,10], tau=4.0, w=0.006):
    peaks=[]
    for e in E_values:
        syn.e=e; syn.tau=tau; nc.weight[0]=w
        h.finitialize(-65*mV); h.continuerun(35*ms)
        vnp=np.array(vB); tnp=np.array(t)
        peaks.append((e, vnp.max()))
        plt.figure(figsize=(5.4,2.6)); plt.plot(tnp,vnp,'k'); plt.axhline(-65,color='gray',ls='--',lw=0.8)
        plt.title(f'E4: E_syn={e} mV'); plt.xlabel('Time (ms)'); plt.ylabel('mV'); plt.grid(True,alpha=0.3); plt.tight_layout(); plt.show()
    return peaks
peaks=reversal_sweep(); print('Peak Vm by E_syn (mV):',peaks)


## E5 — Temporal summation (two spikes, variable ISI) <a id='e5'></a>

In [ ]:

# temporal summation with NetStim

def temporal_sum(isi=5.0,e_rev=0.0,tau=2.0,w=0.005,start=5.0):
    syn.e=e_rev; syn.tau=tau
    ns=h.NetStim(); ns.number=2; ns.start=start; ns.interval=isi; ns.noise=0
    nc_ns=h.NetCon(ns,syn); nc_ns.weight[0]=w
    h.finitialize(-65*mV); h.continuerun((start+isi+15)*ms)
    plot_vm(t,[vB],["Neuron B"],title=f'E5: Temporal summation (ISI={isi} ms)')
for isi in [2.0,5.0,20.0]: temporal_sum(isi=isi)


## E6 — Spatial summation (proximal vs distal dendritic synapses) <a id='e6'></a>

In [ ]:

# add dendrite to B and compare proximal vs distal
try:
    dendB
except NameError:
    dendB=h.Section(name='dendB'); dendB.L=300; dendB.diam=1.5; dendB.Ra=100; dendB.cm=1; dendB.insert('pas'); dendB.connect(somaB(1.0))

synP=h.ExpSyn(dendB(0.2)); synP.tau=2.0; synP.e=0.0
synD=h.ExpSyn(dendB(0.8)); synD.tau=2.0; synD.e=0.0
nsP=h.NetStim(); nsP.number=1; nsP.start=6.0; nsP.noise=0
nsD=h.NetStim(); nsD.number=1; nsD.start=6.0; nsD.noise=0
ncP=h.NetCon(nsP,synP); ncP.weight[0]=0.005
ncD=h.NetCon(nsD,synD); ncD.weight[0]=0.005
v_soma=mk_rec(somaB); v_prox=h.Vector().record(dendB(0.2)._ref_v); v_dist=h.Vector().record(dendB(0.8)._ref_v)

h.finitialize(-65*mV); h.continuerun(25*ms)
plt.figure(figsize=(7,3.3))
plt.plot(t,v_prox,label='dend(0.2)'); plt.plot(t,v_dist,label='dend(0.8)'); plt.plot(t,v_soma,label='soma',color='k')
plt.xlabel('Time (ms)'); plt.ylabel('mV'); plt.title('E6: Spatial summation & attenuation'); plt.grid(True,alpha=0.3); plt.legend(); plt.tight_layout(); plt.show()


## E7 — Feedforward inhibition (E then I) <a id='e7'></a>

In [ ]:

synE2=h.ExpSyn(somaB(0.5)); synE2.tau=2.0; synE2.e=0.0
synI2=h.ExpSyn(somaB(0.5)); synI2.tau=8.0; synI2.e=-75.0
nsE=h.NetStim(); nsE.number=1; nsE.start=6.0; nsE.noise=0
nsI=h.NetStim(); nsI.number=1; nsI.start=8.0; nsI.noise=0
ncE2=h.NetCon(nsE,synE2); ncE2.weight[0]=0.006
ncI2=h.NetCon(nsI,synI2); ncI2.weight[0]=0.01
v_ff=mk_rec(somaB)
h.finitialize(-65*mV); h.continuerun(25*ms)
plot_vm(t,[v_ff],["Neuron B"],title='E7: Feedforward inhibition (E→I)')


## E8 — Analysis helpers (latency, peak, area) <a id='e8'></a>

In [ ]:

import numpy as np

def psp_metrics(tvec,vtrace,baseline=(0,5)):
    tnp=np.array(tvec); vnp=np.array(vtrace)
    b=(tnp>=baseline[0])&(tnp<=baseline[1]); v0=vnp[b].mean() if b.any() else vnp[0]
    peak=vnp.max(); tpk=tnp[vnp.argmax()]
    area=np.trapz(vnp-v0,tnp)
    return {'V0':v0,'Vpeak':peak,'tpeak_ms':tpk,'Area_mV_ms':area}
print('E8 metrics (last trace on vB):', psp_metrics(t,vB))


## E9 — Playground (tweak synaptic parameters and rerun) <a id='e9'></a>

In [ ]:

syn.e=0.0; syn.tau=2.0; nc.weight[0]=0.008; nc.delay=0.5
h.finitialize(-65*mV); h.continuerun(30*ms)
plot_vm(t,[vA,vB],["Neuron A","Neuron B"],title='E9: Playground run')


## Reflection <a id='reflection'></a>
- How do E_syn and τ jointly control PSP amplitude and timing?
- When does inhibitory input look depolarizing yet stay functionally inhibitory?
- How do ISI and electrotonic distance interact in postsynaptic integration?

## Practice / Discussion Questions — Set B — Biology → Model Mapping (20)

1) Define a **model abstraction** for a neuron that preserves interpretability: which parameters must be in **biophysical units**, and why?
2) Explain how **blocks** (RC membrane, synaptic conductance, spike generator) can be composed into a testable single‑cell model.
3) *Justify*: What’s gained and lost when moving from detailed HH‑type to reduced spiking models?
4) How do you **document assumptions** so a reader can reproduce and critique your model? (List 3–4 items.)
5) What’s a **sanity check** for a synapse implemented with a fixed reversal potential? (Describe the test and expected result.)
6) Outline a minimal **reproducible workflow** (notebook → plots → metrics) for a membrane step test.
7) Why is **parameter transparency** essential for educational modeling? Give one consequence of opaque scaling.
8) *Design*: Write 3 summary metrics you would log for “EPSP → spike coupling” experiments.
9) Choose one **modeling decision** (e.g., omit Ca²⁺) and argue when it’s acceptable for instruction.
10) What is the **role of NEURON** in enabling reproducibility for single cells and small networks?
11) *Compare*: When do you favor **current clamp** vs **conductance injection** to test a component?
12) Define a **minimal dendrite** you would include to capture location effects without overcomplicating the model.
13) What’s the educational **trade‑off** of adding noise sources to a beginner model?
14) How would you **validate** your inhibitory synapse timing motif before using it in a network?
15) *Scenario*: Your EPSPs look too broad. List 3 plausible causes and the diagnostic you’d run.
16) Why do we advocate **fixed random seeds** in notebooks for teaching?
17) Provide a **short rubric** for students to self‑assess whether their model description is reproducible by others.
18) Suggest two **figures** (plots/tables) that best communicate a membrane and synapse validation.
19) Give an example of a **missing mechanism** signaled by a consistent mismatch between model and data.
20) *Reflect*: What will you carry from Set B into Set C–E when you begin analyzing spatial effects?



---

## Practice / Discussion Questions — Set E — Synapses & Transmission (20)

1) Explain how **fixed reversal potentials** make synaptic efficacy **state‑dependent**.
2) Define and compare **AMPA‑like** vs **GABA_A** synapses in terms of E_rev and kinetics.
3) *Predict*: How will holding at −50 vs −70 mV change the apparent sign/magnitude of a GABA_A PSP?
4) Describe the **E→I feedforward motif** and its effect on graded→spike conversion.
5) Define a **timing window** (E leads I by Δt) and predict effects on peak and latency.
6) Give one **temporal** and one **spatial** summation example; state what changes.
7) *Reasoning*: Why do distal excitatory inputs often require cooperation to impact the soma?
8) What does **shunting inhibition** mean and how is it revealed experimentally?
9) Name two **metrics** to evaluate strength/timing of synaptic integration.
10) Sketch a protocol to verify that E→I reduces spike probability without changing E magnitude.
11) How does membrane **state** (input resistance, V_m) gate synaptic impact?
12) Distinguish **incidence of local events** vs **coupling** to soma; give an example.
13) *Critique*: Risk of using only somatic measures for synaptic efficacy.
14) Why does inhibitory **location** (perisomatic vs dendritic) matter for timing and gain?
15) Provide a case where inhibitory timing **facilitates** information transmission; explain.
16) *Compute/Explain*: How does driving force change as V_m approaches E_rev for an inhibitory synapse?
17) Quick **two‑condition test** to separate shunting from hyperpolarizing inhibition.
18) If two EPSPs overlap, when expect **superlinear** vs **sublinear** summation? Why?
19) Connect Set E outcomes to **network persistence** (why timing matters).
20) State the minimal integration insights from E you’ll carry to Set F.
